In [1]:
import pandas as pd
import pylab as pl
import numpy as np
import scipy.optimize as opt
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
os.chdir(r"C:\Users\cebal\OneDrive\Documentos\Grandes Bases de Datos I")
cell_df = pd.read_csv("recursos_humanos (1).csv")
cell_df

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,sales,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14994,0.40,0.57,2,151,3,0,1,0,support,low
14995,0.37,0.48,2,160,3,0,1,0,support,low
14996,0.37,0.53,2,143,3,0,1,0,support,low
14997,0.11,0.96,6,280,4,0,1,0,support,low


In [ ]:
cell_df["Class"] = cell_df["Class"].replace([2], 0)  # Tumor Benigno
cell_df["Class"] = cell_df["Class"].replace([4], 1)  # Tumor Maligno
cell_df

In [ ]:
# Diagrama de dispersión
ax = cell_df[cell_df["Class"] == 1][0:50].plot(
    kind="scatter", x="Clump", y="Unfsize",
    color="DarkBlue", label="Maligno"
)
ax = cell_df[cell_df["Class"] == 0][0:50].plot(
    kind="scatter", x="Clump", y="Unfsize",
    color="Yellow", label="Benigno", ax= ax
)
plt.show()

In [ ]:
# Pre-Procesamiento de datos y selección
cell_df.dtypes

In [ ]:
# Eliminación de renglones no numéricos en BareNuc
cell_df = cell_df[pd.to_numeric(cell_df["BareNuc"], errors="coerce").notnull()]
cell_df["BareNuc"] = cell_df["BareNuc"].astype("int")
cell_df.dtypes

In [ ]:
feature_df = cell_df[["Clump", "UnifSize", "UnifShape", "MargAdh",
                      "SingEpiSize", "BareNuc", "BlandChrom",
                      "NormNucl", "Mit"]]
x = np.asarray(feature_df)
x[0:5]

In [ ]:
cell_df["Class"] = cell_df["Class"].astype("int")
y = np.asarray(cell_df["Class"])
y[0:5]

In [ ]:
# Creación de grupos de entrenamiento y prueba
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=4
)
print("Grupo de entrenamiento", x_train.shape, y_train.shape)
print("Grupo de prueba", x_test.shape, y_test.shape)

Modelacion con SVM
Opciones de Kernel (Transformaciones):
1. Linear
2. Polynomial
3. Rdial Basis Function (RBF)
4. Sigmoid

In [ ]:
from sklearn import svm
clf = svm.SVC(kernel="rbf")
clf.fit(X_train, y_train)

In [ ]:
# Predicción para la base de prueba
yhat = clf.predict(X_test)
yhat[0:5]

In [ ]:
# Creación de matriz de confusión
from sklearn.metrics import confusion_matrix
y_pred = clf.predict(X_test)
cm = confusion_matrix(y_test, yhat)
cm

In [ ]:
y_test

In [ ]:
yhat

In [ ]:
z = y_test - yhat
z

In [ ]:
# Visualización de la matriz de confusión
import seaborn as sns
f, ax = plt.subplots(figsize=(5, 5))
sns.heatmap(cm, annot=True, linewidths=0.5,
            linecolor="red", fmt=".0f", ax=ax)
plt.xlabel("Y pronosticada")
plt.ylabel("Y real")
plt.show()

Estadisticas de Desempeño

In [ ]:
from sklearn.metrics import classification_report
cnf_matrix = confusion_matrix(y_test, yhat, labels=[0,1])
print(classification_report(y_test, yhat))

 Determinacion de Nivelies de Precision
 La Precision = Porcentaje de predicciones correctas relativas al total de predicciones
 El total calcular por columna
 se contesta la pregunta: ¿que porcentake de las veces que hacemos un pronostico de cierto tipo este correcto?


In [ ]:
PrecisionBenigno = cm[0,0] / (cm[0,0] + cm[1,0])
PrecisionBenigno

In [ ]:
PrecisionMaligno = cm[1,1] / (cm[1,1] + cm[0,1])
PrecisionMaligno

Determinacion de Niveles de Recuperacion (Recall)
la recuperacion (recall) = Porcentaje de predicciones correctas relativas al total de valores reales
el total se calcula por renglon
¿que porcentaje de las veces se tiene un valor identico?

In [ ]:
RecallBenigno = cm[0,0] / (cm[0,0] + cm[0,1])
RecallBenigno

In [ ]:
RecallMaligno = cm[1,1] / (cm[1,0] + cm[1,1])
RecallMaligno

Determinacion del Score F1
F1 score = media armonica ponderada de la presicion y la recuperacion. Cuanto mas se acerque a 1, mejor sera el modelo

F1 Score = 2*(Precison * Recall)/(Precision + Recall)

In [ ]:
F1Benigno = 2 * (PrecisionBenigno * RecallBenigno) / (PrecisionBenigno + RecallBenigno)
F1Benigno

F1Maligno = 2 * (PrecisionMaligno * RecallMaligno) / (PrecisionMaligno + RecallMaligno)
F1Maligno

Determinacion de soportes
Soporte ) Numero de observaciones que permanecen en forma real a cada clase posible (total por renglon)

In [ ]:
SoporteBenigno = cm[0,0] + cm[0,1]
SoporteBenigno

SoporteMaligno = cm[1,0] + cm[1,1]
SoporteMaligno

Determinacion de la precision Global
Accuracy = Porcentaje de predicciones correctas

In [ ]:
Correctos = cm[0,0] + cm[1,1]
Incorrectos = cm[0,1] + cm[1,0]
PrecisionGlobal = Correctos / (Correctos + Incorrectos)
PrecisionGlobal

Promedios simples por indicador (Macro Average)
Promedio simples por precision, recuperacion y f1-score

In [ ]:
MacroAvgPrecision = (PrecisionBenigno + PrecisionMaligno) / 2
MacroAvgPrecision

MacroAvgRecall = (RecallBenigno + RecallMaligno) / 2
MacroAvgRecall

MacroF1Score = (F1Benigno + F1Maligno) / 2
MacroF1Score

Promedios ponderados por indicador (Weighted Average)
Promedios ponderados de acuerdo a soportes por indicador

In [ ]:
PesoBenigno = SoporteBenigno / (SoporteBenigno + SoporteMaligno)
PesoMaligno = SoporteMaligno / (SoporteBenigno + SoporteMaligno)

WAvgPrecision = PrecisionBenigno * PesoBenigno + PrecisionMaligno * PesoMaligno
WAvgPrecision

WAvgRecall = RecallBenigno * PesoBenigno + RecallMaligno * PesoMaligno
WAvgRecall

WAvgF1Score = F1Benigno * PesoBenigno + F1Maligno * PesoMaligno
WAvgF1Score

Prueba de SVM con Kernel Lineal

In [ ]:
clf = svm.SVC(kernel='linear')
clf.fit(X_train, y_train)

yhat = clf.predict(X_test)

# Confusion matrix
from sklearn.metrics import confusion_matrix
y_pred = clf.predict(X)
cm = confusion_matrix(y_test, yhat)
cm

In [ ]:
from sklearn.metrics import classification_report

# Importar iterools
cnf_matrix = confusion_matrix(y_test, yhat, labels=[0,1])
# np.set_printoptions(precision=2)
print(classification_report(y_test, yhat))

Prueba de SVM con Kernel Polinomial

In [ ]:
clf = svm.SVC(kernel='poly')
clf.fit(X_train, y_train)
yhat = clf.predict(X_test)

# Confusion matrix
from sklearn.metrics import confusion_matrix
y_pred = clf.predict(X)
cm = confusion_matrix(y_test, yhat)
cm

In [ ]:
from sklearn.metrics import classification_report
# import itertools
cnf_matrix = confusion_matrix(y_test, yhat, labels=[0,1])
# np.set_printoptions(precision=2)
print(classification_report(y_test, yhat))

Prueba de SVM con Kernel Sigmoide

In [ ]:
clf = svm.SVC(kernel='poly')
clf.fit(X_train, y_train)
yhat = clf.predict(X_test)
# Confusion matrix
from sklearn.metrics import confusion_matrix
y_pred = clf.predict(X)
cm = confusion_matrix(y_test, yhat)
cm

In [ ]:
from sklearn.metrics import classification_report
# import itertools
cnf_matrix = confusion_matrix(y_test, yhat, labels=[0,1])
# np.set_printoptions(precision=2)
print(classification_report(y_test, yhat))

Prueba de SVM con Kernel Sifmoide

In [ ]:
clf = svm.SVC(kernel='sigmoid')
clf.fit(X_train, y_train)
yhat = clf.predict(X_test)
# Confusion matrix
from sklearn.metrics import confusion_matrix
y_pred = clf.predict(X)
cm = confusion_matrix(y_test, yhat)
cm

In [ ]:
from sklearn.metrics import classification_report
# import itertools
cnf_matrix = confusion_matrix(y_test, yhat, labels=[0,1])
# np.set_printoptions(precision=2)
print(classification_report(y_test, yhat))